# Project 1: A marketing agent that learns from its customers

You work in the marketing team of an online clothing shop with 700 customers. Every week the shop sends an email. Whether a customer opens the shop or ignores the email depends mostly on its **first line, the "hook"**.

In this project you build a small **AI marketing agent** that writes hooks and improves them with **OPRO** (*Optimization by PROmpting*): a language model looks at the hooks you sent before and how well they did, and proposes a better one. You never write a formula. You steer the agent with the instructions you give it, the **meta-prompt**.

**The twist:** the customers are simulated. For each customer group, a language model plays a customer with a **hidden description** of what this customer likes and dislikes. You never see that description. Like a real marketing team, you only see how many customers visited the shop, and you have some customer feedback to learn from.

### How this notebook works
- Run the cells from top to bottom with **Shift + Enter**.
- Cells marked **🎯 TODO** are your work: 9 in total. After most of them, a **check cell** prints ✅ or ❌.
- Cells that take longer than a minute say so with ⏱️.
- First: **File → Save a copy in Drive**, so your work is kept.

| Part | What you do | TODOs |
|---|---|---|
| 0 | Set up the notebook and load the language model | |
| 1 | Meet your customers: look at the data, write hooks by hand and see how they react | 1 |
| 2 | Sort the customers into segments | 2, 3 |
| 3 | Simulate an email campaign | 4, 5 |
| 4 | Build the pieces of the OPRO agent | 6, 7 |
| 5 | Run OPRO, compare, experiment | 8, 9 |

## Part 0: Setup

**Runtime → Change runtime type → T4 GPU**, then run the two cells below. The first one fetches the project files and the helper functions in `marketing.py` (you can open that file and read it, but you do not need to change it). The second one loads the language model.

In [ ]:
# Get the project files and the helper functions
import os
REPO_URL = "https://github.com/eth-ainit-hs26/w1-project-public.git" 
!git clone -q {REPO_URL} project1
os.chdir("project1")
import marketing
import checks

print("Ready. Working folder:", os.getcwd())

Load the language model. It downloads about 10 GB the first time (under a minute on Colab).

In [ ]:
marketing.setup_model()

## Part 1: Meet your customers

The customer data is in `data/customers.csv`. `marketing.load_customers()` reads it into a **list of dictionaries**, one dictionary per customer:

In [ ]:
customers = marketing.load_customers()
print(len(customers), "customers. The first three:")
for customer in customers[:3]:
    print(customer)

Every customer has:
- `n_purchases`: number of orders so far
- `avg_order_value`: average amount per order, in CHF
- `days_since_last_purchase`: days since the last order (`None` if the customer never ordered)
- `satisfaction`: their last satisfaction rating between 0 and 1 (`None` if they never ordered)

A list of 700 dictionaries is hard to read. A **histogram** is easier: for each column, it shows how many customers have which value. The taller the bar, the more customers have that value.

In [ ]:
marketing.plot_customer_data(customers)

*Think about it (no need to write anything):* Are most customers young or old? Do most order often or rarely? Look at the number of orders: one group of customers has never ordered anything. Some customers have not ordered for a long time, or were not happy with their last order. Groups like these are what the marketing team calls segments.

### The five segments

The shop sorts its customers into five **segments**:

| Letter | Segment |
|---|---|
| N | New customers |
| H | High-value customers |
| R | Regular customers |
| L | Low-budget customers |
| A | At-risk customers |

The function `marketing.customer_reaction(segment, hook)` shows a hook to the hidden customer of a segment and returns a **score between 0 and 1**: how much this customer wants to click. Behind it, the language model plays the customer, reads the hook and answers "Would you visit the shop?". For you, it is a **black box**: text goes in, a number comes out.

Here is how every segment reacts to the hook the shop has been using so far:

In [ ]:
for segment in marketing.SEGMENTS:
    score = marketing.customer_reaction(segment, marketing.STARTING_HOOK)
    print(segment, round(score, 2), marketing.STARTING_HOOK)

### 🎯 TODO 1: Write hooks by hand

Choose one segment and write **three different hooks** for it. Try three quite different ideas (for example a discount, a personal tone, news about the collection). Run the cell to see the scores.

In [ ]:
# 🎯 TODO 1: choose a segment and write three different hooks for it
my_segment = "L"     # one of "N", "H", "R", "L", "A"
my_hooks = [
    "...",   # replace ... with your first hook
    "...",   # your second hook
    "...",   # your third hook
]

scores = []
for hook in my_hooks:
    score = marketing.customer_reaction(my_segment, hook)
    scores.append(score)
    print(round(score, 2), hook)

checks.check_my_hooks(my_hooks, scores)

**🎯 TODO 1 (continued):** Which hook scored best? What do you guess this customer cares about? Write two or three sentences here.

*Your answer:*

## Part 2: Sort the customers into segments

Now give every customer from Part 1 a segment. The marketing team uses these rules. **Check them in this order** and stop at the first one that fits:

| # | Rule | Segment |
|---|---|---|
| 1 | no purchases at all | `"N"` |
| 2 | at least 4 purchases **and** (satisfaction at most 0.45 **or** last purchase more than 60 days ago) | `"A"` |
| 3 | at least 5 purchases **and** last purchase at most 90 days ago **and** average order value at least CHF 45 | `"H"` |
| 4 | average order value at least CHF 15 | `"R"` |
| 5 | everybody else | `"L"` |

### 🎯 TODO 2: `assign_segment(customer)`
Write a function that returns the segment letter for one customer. Use `if` and `return` for each rule. A customer's values are read like `customer["n_purchases"]`.

In [ ]:
def assign_segment(customer):
    """Return the segment letter ("N", "A", "H", "R" or "L") for one customer."""
    # 🎯 TODO 2: write the five rules from the table, in the same order.
    # Example of the first rule:
    #     if customer["n_purchases"] == 0:
    #         return "N"
    return "L"   # placeholder: replace this line with your rules


checks.check_assign_segment(assign_segment)

Now give every customer their segment. This adds a new key `"segment"` to each customer dictionary:

In [ ]:
for customer in customers:
    customer["segment"] = assign_segment(customer)

print(customers[0])

### 🎯 TODO 3: `count_per_segment(customers)`
How big is each segment? Write a function that returns a dictionary like `{"N": 130, "H": 130, ...}`.

*Hint:* start with an empty dictionary, loop over the customers, and add 1 to the count of each customer's segment. `counts.get(segment, 0)` gives the current count, or 0 if there is none yet.

In [ ]:
def count_per_segment(customers):
    """Return a dictionary: segment letter -> number of customers in that segment."""
    counts = {}
    # 🎯 TODO 3: loop over the customers and count them per segment
    return counts


checks.check_count_per_segment(count_per_segment)
print("Our customers:", count_per_segment(customers))

Once we have segmented our customers, we can visualize the distributions of per-segment customer features using box plots:

In [ ]:
marketing.plot_segments(customers)

*Sanity Check:* Are the distributions in the visualization consistent with the rules from the table? If not, check your `assign_segment` again.

Besides the numbers, the marketing team has a short **description** of each segment and some **customer feedback** from surveys and support tickets. Read it carefully: it is the only information you have about what these customers want. (It may also contain things that do not matter.)

In [ ]:
segments = marketing.load_segments()
feedback = marketing.load_feedback()

for segment in marketing.SEGMENTS:
    print(f"{segment} - {segments[segment]['name']}: {segments[segment]['description']}")
    for quote in feedback[segment]:
        print(f'      "{quote}"')
    print()

## Part 3: Simulate an email campaign

A score is not yet a visit. Real customers are unpredictable: even a great email does not convince everybody, and some people visit the shop anyway. So in our simulation every customer **flips a biased coin**. The chance to visit is

> visit probability = base rate + lift × score

with **base rate 0.05** (5% visit no matter what) and **lift 0.6** (a perfect hook adds 60 percentage points).

### 🎯 TODO 4: `visit_probability(score, base_rate, lift)`

In [ ]:
def visit_probability(score, base_rate, lift):
    """The chance that a customer visits the shop after reading a hook with this score."""
    # 🎯 TODO 4: return base_rate + lift * score
    return 0


checks.check_visit_probability(visit_probability)

`marketing.simulate_campaign(customers, hook_per_segment, visit_probability)` sends one email campaign: every customer gets the hook of their segment, and the function returns a dictionary `{customer_id: True or False}` saying who visited the shop. It uses your `visit_probability` function for the coin flips.

### 🎯 TODO 5: `visit_rate_per_segment(customers, visited)`
Turn that into the number the marketing team cares about: **what share of each segment visited the shop?** Return a dictionary like `{"N": 0.16, "H": 0.19, ...}`.

*Hint:* count two things per segment: how many customers there are, and how many of them visited (`visited[customer["customer_id"]]` is `True` or `False`). Then divide.

In [ ]:
def visit_rate_per_segment(customers, visited):
    """Return a dictionary: segment letter -> share of its customers who visited (between 0 and 1)."""
    # 🎯 TODO 5: count customers and visits per segment, then divide
    rates = {}
    return rates


checks.check_visit_rate_per_segment(visit_rate_per_segment)

Now send the current hook to everybody, not once but **30 times**, and look at the visit rate of each campaign. Each bar counts how many of the 30 campaigns got that visit rate:

In [ ]:
current_hooks = {segment: marketing.STARTING_HOOK for segment in marketing.SEGMENTS}

marketing.plot_campaign_noise(customers, current_hooks, visit_probability, visit_rate_per_segment)

*Think about it (no need to write anything):* the same hook, sent to the same customers, gives a different visit rate every time. Why? How wide is the spread in each segment? What does that mean when you compare two hooks whose visit rates are only a few percent apart?

## Part 4: Build the OPRO agent

OPRO repeats one loop, separately for every segment:

```
   ┌──────────────────────────────────────────────────────────────┐
   │ history: the hooks we sent so far and their visit rates      │
   └──────────────┬───────────────────────────────────────────────┘
                  │ 1. build_meta_prompt(segment, history, k)   ← TODO 6 + 7
                  ▼
          meta-prompt ──► language model ──► answer
                                                │ 2. clean_hook(answer)
                                                ▼
                                            new hook ──► campaign ──► visit rate
                                                                          │ 3. add to the history
                  ┌───────────────────────────────────────────────────────┘
                  ▼
               next round
```

The history of one segment is a list of dictionaries, for example:

```python
[
    {"round": 0, "hook": "Check out our latest offers!", "visit_rate": 0.21, "score": 0.27},
    {"round": 1, "hook": "Free shipping on all orders this week.", "visit_rate": 0.36, "score": 0.52},
]
```

### 🎯 TODO 6: `format_history(history, k)`
The language model can only learn from the history if it can read it. Write a function that turns the **k best attempts** (highest `visit_rate`) into text, **sorted from worst to best** (the best one last, where the model pays most attention), with the visit rate as a percentage. For example:

```
Hook: "Check out our latest offers!"
Visit rate: 21%

Hook: "Free shipping on all orders this week."
Visit rate: 36%
```

In [ ]:
def get_visit_rate(attempt):
    return attempt["visit_rate"]

def format_history(history, k):
    """The k best attempts as text, sorted from worst to best."""
    # 🎯 TODO 6: sort, reverse, keep the k best, and build the text
    best_first = sorted(history, key=get_visit_rate, reverse=...)
    k_best = ... # first k elements of best_first
    parts = [] # build a list of strings, one for each attempt
    for attempt in reversed(k_best):
        # the hook and visit rate of this attempt, formatted as in the example
        attempt_text = ... # Hint: use f-strings and the format specifier .0% to format the visit rate as a percentage
        parts.append(attempt_text)    
    text = "\n\n".join(parts)
    return text


checks.check_format_history(format_history)

### Helper function: `clean_hook(text)`
Language models do not always answer with just the hook. They add quotes, line breaks, or a short explanation first. The function `clean_hook(text)`  picks out the hook from a model output with the following rules:

1. If the answer has a line that starts with `Hook:`, keep only the text after `Hook:` on that line (useful if your meta-prompt asks the model to think first).
2. Otherwise take the first line that is not empty.
3. Remove spaces and quote marks (`"`) at both ends.
4. If the hook still contains `[` (a placeholder like `[Name]`), return `""`, so the hook is thrown away.

In [ ]:
def clean_hook(text):
    """Pick the hook out of the language model's answer. Return "" if there is no usable hook."""
    lines = []
    for line in text.splitlines():
        if line.strip() != "":
            lines.append(line.strip())
    if len(lines) == 0:
        return ""
    hook = lines[0]
    for line in lines:
        if line.startswith("Hook:"):
            hook = line[5:]
            break
    hook = hook.strip().strip('"').strip()
    if "[" in hook:
        return ""
    return hook

### 🎯 TODO 7: `build_meta_prompt(segment, history, k)`: the main task

The **meta-prompt** is the instruction that tells the language model how to write the next hook. It decides what the model knows and how it thinks, so it decides how good your agent is. The version below works, but it is very basic. **Improve it.** Ideas:

- Give the model a **role** and explain the situation (the shop, what a hook is).
- Tell it **who the customers are**: `segments[segment]["description"]`.
- Give it the **customer feedback**: `feedback[segment]` is a list of quotes. It is the only real clue about what customers want.
- Show the history with your `format_history(history, k)` and say what the numbers mean.
- Ask it to **think first**: what do these customers care about, what stops them from buying, what made the best hooks better than the worst? If you do, ask it to put the hook on its own line starting with `Hook:`, so your `clean_hook` finds it.
- Ask for something **new and concrete** instead of a small variation of the best hook.
- Set **rules**: length, no placeholders like `[Name]`.

There is no single right answer. Different meta-prompts lead to different results, and you will explain your choices in the defence.

In [ ]:
def build_meta_prompt(segment, history, k):
    """The instruction for the language model: write a better hook for this segment."""
    # Loading customer segment description and feedback quotes to include in your meta-prompt
    description = segments[segment]["description"]
    quotes = "" 
    for quote in feedback[segment]:
        quotes = quotes + f'- "{quote}"\n'
    
    # Loading the k best past hooks
    past_hooks = format_history(history, k)
    
    # 🎯 TODO 7: improve this meta-prompt (see the ideas above)
    prompt = f"""Write a marketing email hook.

Past hooks:
{past_hooks}

Reply with the hook only."""
    return prompt


checks.check_meta_prompt(build_meta_prompt)

Before running the whole loop, try your meta-prompt once: here it writes three hooks for the at-risk customers (A), starting from the current hook. For each answer you see the hook, the model's thinking (everything else it wrote) and how the hidden customer reacts. Reading the thinking shows you whether the model understood your meta-prompt.

In [ ]:
first_history = [{"round": 0, "hook": marketing.STARTING_HOOK, "visit_rate": 0.21, "score": 0.27}]
test_prompt = build_meta_prompt("A", first_history, 8)
answers = marketing.ask_llm_many([test_prompt], n=3)[0]

for answer in answers:
    hook = clean_hook(answer)

    # The thinking is every other line of the answer, without the "Thinking:" label.
    thinking_lines = []
    for line in answer.splitlines():
        line = line.strip()
        if line == "" or line.startswith("Hook:") or (hook != "" and hook in line):
            continue
        if line.startswith("Thinking:"):
            line = line[9:].strip()
        thinking_lines.append(line)
    thinking = " ".join(thinking_lines)

    if hook == "":
        print("Hook:     (no usable hook in this answer)")
    else:
        print("Hook:    ", hook)
    if thinking == "":
        print("Thinking: (none: your meta-prompt does not ask the model to think first)")
    else:
        print("Thinking:", thinking)
    if hook != "":
        print("Reaction:", round(marketing.customer_reaction("A", hook), 2))
    print()

## Part 5: Run OPRO

`marketing.run_opro(...)` runs the loop for all five segments at once. Each round, the model writes 4 new hooks per segment from the meta-prompt, every new hook is sent in a simulated campaign, and the results go into the history. You give it your functions by name, without brackets.

### 🎯 TODO 8: Run OPRO and compare with the baseline
First run the **baseline**, a simple meta-prompt from `marketing.py`, then your own. ⏱️ The baseline takes about 2 min on a Colab T4. Your own run takes longer, about 4.5 min for the example solution, because a meta-prompt that asks the model to think first makes it write more.

In [ ]:
baseline_run = marketing.run_opro(
    customers, marketing.baseline_meta_prompt, clean_hook, visit_probability, visit_rate_per_segment,
    label="baseline",
)

In [ ]:
my_run = marketing.run_opro(
    customers, build_meta_prompt, clean_hook, visit_probability, visit_rate_per_segment,
    label="my meta-prompt",
)

In [ ]:
marketing.plot_runs([baseline_run, my_run])
marketing.show_best_hooks(my_run)
marketing.compare_runs([baseline_run, my_run])

The plot above only shows the **best** hook so far, so its line can never go down. Here is everything your run tried. Each grey dot is one hook and the visit rate it got. The blue line is the average of each round, and the orange dashed line is the best so far from above.

In [ ]:
marketing.plot_all_hooks(my_run)

And here is each hook's score from the hidden customer against the visit rate it actually got. Without the coin flips, every dot would lie on the dashed line.

In [ ]:
marketing.plot_score_vs_visit_rate(my_run)

*Think about it (no need to write anything):* Does the whole round get better over time, or only the best hook? How far are the dots from the dashed line? The best hook so far was probably lucky as well as good. Would it do as well if you sent it again?

**🎯 TODO 8 (continued):** Describe what you see in three to five sentences. Which segments improved and which did not? In which round did most of the improvement happen? Did your meta-prompt beat the baseline, and in which segments? Look at the best hooks: why might they work for these customers?

*Your answer:*

### 🎯 TODO 9: Your experiment
Change **one thing**, run again, and compare with your first run. Changing only one thing at a time is how you find out what actually made the difference. Ideas:

- Remove the customer feedback from your meta-prompt, or remove the "think first" part. How much did it contribute?
- Show fewer or more past hooks: `k=3` or `k=15` in `run_opro(...)`.
- Make the model more or less creative: `temperature=0.5` or `temperature=1.3`.
- Try more candidates per round with fewer rounds: `candidates=8, rounds=3`.

Write your changed meta-prompt as a new function (for example `build_meta_prompt_v2`) so your first version stays as it is. ⏱️ About 4.5 min on a Colab T4 with the default settings (measured with the example solution). Feel free to experiment with multiple versions.

In [ ]:
# 🎯 TODO 9: change one thing at a time and run again
def build_meta_prompt_v2(segment, history, k):
    prompt = "your new meta-prompt here"
    return prompt

In [ ]:
experiment_run = marketing.run_opro(
    customers, build_meta_prompt_v2, clean_hook, visit_probability, visit_rate_per_segment,
    label="experiment: ...",   # describe your change here
)

marketing.plot_runs([my_run, experiment_run])
marketing.compare_runs([baseline_run, my_run, experiment_run])

**🎯 TODO 9 (continued):** What did you change, what did you expect, and what happened? Was the difference larger than the noise you saw in Part 3?

*Your answer:*

## What to hand in
Save this notebook with all outputs (**File → Download → .ipynb**) and hand it in as instructed by the course team.